ENERGY CSV DATA CREATION FOR OUR PROJECT


In [ ]:
import pandas as pd
import random
from datetime import datetime

# --- BUSINESS CATEGORIES WITH ESTIMATED kWh RANGES ---
businesses = {
    "Bakery": {"range": (1200, 2500)},
    "Supermarket": {"range": (3000, 8000)},
    "SmallShop": {"range": (500, 1200)},
    "Restaurant": {"range": (1500, 4000)},
    "Pharmacy": {"range": (800, 2000)},
    "FuelStation": {"range": (5000, 12000)}  # fuel pumps + lighting + office
}

n_months = 12   # 12 months of data (1 year)
start_date = datetime(2024, 1, 1)

rows = []
business_id = 1

# --- GENERATE SYNTHETIC DATA ---
for biz_type, values in businesses.items():
    for i in range(10):  # 10 businesses per type
        current_date = start_date
        for m in range(n_months):
            billing_period = current_date.strftime("%Y-%m")

            # Grid usage
            grid_kwh = random.randint(values["range"][0], values["range"][1])
            rows.append([f"B{business_id:04}", biz_type, billing_period, grid_kwh, "grid"])

            # Generator usage (smaller portion, but higher in Nigeria context)
            gen_kwh = random.randint(int(grid_kwh * 0.1), int(grid_kwh * 0.4))
            rows.append([f"B{business_id:04}", biz_type, billing_period, gen_kwh, "generator"])

            # Move to next month
            next_month = current_date.month % 12 + 1
            next_year = current_date.year + (current_date.month // 12)
            current_date = datetime(next_year, next_month, 1)

        business_id += 1

# --- CREATE DATAFRAME ---
df = pd.DataFrame(rows, columns=["business_id", "business_type", "billing_period", "consumption_kWh", "source"])

# --- SAVE TO CSV ---
df.to_csv("energy.csv", index=False)

print("Synthetic energy.csv created with", len(df), "rows")
print(df.sample(10))


Synthetic energy.csv created with 1440 rows
     business_id business_type billing_period  consumption_kWh     source
760        B0032    Restaurant        2024-09             2597       grid
540        B0023     SmallShop        2024-07             1112       grid
904        B0038    Restaurant        2024-09             3788       grid
46         B0002        Bakery        2024-12             2090       grid
611        B0026     SmallShop        2024-06              332  generator
1042       B0044      Pharmacy        2024-06             1674       grid
735        B0031    Restaurant        2024-08              923  generator
916        B0039    Restaurant        2024-03             3778       grid
838        B0035    Restaurant        2024-12             2395       grid
600        B0026     SmallShop        2024-01              581       grid


In [ ]:
df.head()

,business_id,business_type,billing_period,consumption_kWh,source
0,B0001,Bakery,2024-01,2022,grid
1,B0001,Bakery,2024-01,264,generator
2,B0001,Bakery,2024-02,2152,grid
3,B0001,Bakery,2024-02,544,generator
4,B0001,Bakery,2024-03,2128,grid


In [ ]:
df.shape

(1440, 5)

In [ ]:
df.groupby('consumption_kWh')['source'].value_counts().sort_index(ascending=False)

,,count
consumption_kWh,source,
11958,grid,1
11944,grid,1
11927,grid,1
11881,grid,1
11849,grid,1
...,...,...
70,generator,1
66,generator,1
64,generator,1


In [ ]:
import pandas as pd

# Define emission factors
data = [
    {"activity": "grid_electricity_NG", "unit": "kWh", "emission_factor": 0.508},
    {"activity": "diesel_generator", "unit": "kWh", "emission_factor": 0.75},
    {"activity": "petrol_generator", "unit": "kWh", "emission_factor": 0.72},
]

# Create DataFrame
df2 = pd.DataFrame(data)

# Save to CSV
df2.to_csv("emission_factors.csv", index=False)

print("emission_factors.csv created")
print(df2)


emission_factors.csv created
              activity unit  emission_factor
0  grid_electricity_NG  kWh            0.508
1     diesel_generator  kWh            0.750
2     petrol_generator  kWh            0.720


In [ ]:
df2.head()

,activity,unit,emission_factor
0,grid_electricity_NG,kWh,0.508
1,diesel_generator,kWh,0.750
2,petrol_generator,kWh,0.720


EXTRACT LOAD AND TRANSFORM(ETL)

In [ ]:
import pandas as pd

# --- LOAD DATA ---
df = pd.read_csv("energy.csv")          # usage data
df2 = pd.read_csv("emission_factors.csv")  # emission factors

# --- MAP SOURCES TO FACTOR ACTIVITIES ---
df["activity"] = df["source"].apply(lambda x: "grid_electricity_NG" if x == "grid" else "diesel_generator")

# --- MERGE WITH EMISSION FACTORS ---
df3 = df.merge(df2, how="left", on="activity")

# --- CALCULATE EMISSIONS ---
df3["emissions_kgCO2e"] = df3["consumption_kWh"] * df3["emission_factor"]

# --- SAVE OUTPUT ---
df3.to_csv("emissions_summary.csv", index=False)

print(" emissions_summary.csv created with", len(df3), "rows")
print(df3.head(10))


✅ emissions_summary.csv created with 1440 rows
  business_id business_type billing_period  consumption_kWh     source  \
0       B0001        Bakery        2024-01             2022       grid   
1       B0001        Bakery        2024-01              264  generator   
2       B0001        Bakery        2024-02             2152       grid   
3       B0001        Bakery        2024-02              544  generator   
4       B0001        Bakery        2024-03             2128       grid   
5       B0001        Bakery        2024-03              811  generator   
6       B0001        Bakery        2024-04             2473       grid   
7       B0001        Bakery        2024-04              291  generator   
8       B0001        Bakery        2024-05             2422       grid   
9       B0001        Bakery        2024-05              500  generator   

              activity unit  emission_factor  emissions_kgCO2e  
0  grid_electricity_NG  kWh            0.508          1027.176  
1     di

In [ ]:
df3.head()

,business_id,business_type,billing_period,consumption_kWh,source,activity,unit,emission_factor,emissions_kgCO2e
0,B0001,Bakery,2024-01,2022,grid,grid_electricity_NG,kWh,0.508,1027.176
1,B0001,Bakery,2024-01,264,generator,diesel_generator,kWh,0.750,198.000
2,B0001,Bakery,2024-02,2152,grid,grid_electricity_NG,kWh,0.508,1093.216
3,B0001,Bakery,2024-02,544,generator,diesel_generator,kWh,0.750,408.000
4,B0001,Bakery,2024-03,2128,grid,grid_electricity_NG,kWh,0.508,1081.024


In [ ]:
df3.shape

(1440, 9)

In [ ]:
# --- GROUPED SUMMARY PER BUSINESS ---
business_summary = (
    df3.groupby(["business_id", "business_type"])
       .agg(total_consumption_kWh=("consumption_kWh", "sum"),
            total_emissions_kgCO2e=("emissions_kgCO2e", "sum"))
       .reset_index()
)

business_summary.to_csv("business_summary.csv", index=False)
print("\n business_summary.csv created:")
print(business_summary.head())

# --- GROUPED SUMMARY PER SECTOR (industry type) ---
sector_summary = (
    df3.groupby("business_type")
       .agg(total_consumption_kWh=("consumption_kWh", "sum"),
            total_emissions_kgCO2e=("emissions_kgCO2e", "sum"))
       .reset_index()
)

sector_summary.to_csv("sector_summary.csv", index=False)
print("\n sector_summary.csv created:")
print(sector_summary.head())



 business_summary.csv created:
  business_id business_type  total_consumption_kWh  total_emissions_kgCO2e
0       B0001        Bakery                  28544               15722.452
1       B0002        Bakery                  27704               15221.438
2       B0003        Bakery                  27626               15288.052
3       B0004        Bakery                  29948               16756.092
4       B0005        Bakery                  29520               16380.642

 sector_summary.csv created:
  business_type  total_consumption_kWh  total_emissions_kgCO2e
0        Bakery                 284369              157982.170
1   FuelStation                1299703              725807.650
2      Pharmacy                 212040              118087.956
3    Restaurant                 416799              231915.966
4     SmallShop                 123473               68518.248


In [ ]:
sector_summary.head()

,business_type,total_consumption_kWh,total_emissions_kgCO2e
0,Bakery,284369,157982.170
1,FuelStation,1299703,725807.650
2,Pharmacy,212040,118087.956
3,Restaurant,416799,231915.966
4,SmallShop,123473,68518.248


In [ ]:
 business_summary['business_type'].value_counts()

,count
business_type,
Bakery,10
Supermarket,10
SmallShop,10
Restaurant,10
Pharmacy,10
FuelStation,10


In [ ]:
df3.head()

,business_id,business_type,billing_period,consumption_kWh,source,activity,unit,emission_factor,emissions_kgCO2e
0,B0001,Bakery,2024-01,2022,grid,grid_electricity_NG,kWh,0.508,1027.176
1,B0001,Bakery,2024-01,264,generator,diesel_generator,kWh,0.750,198.000
2,B0001,Bakery,2024-02,2152,grid,grid_electricity_NG,kWh,0.508,1093.216
3,B0001,Bakery,2024-02,544,generator,diesel_generator,kWh,0.750,408.000
4,B0001,Bakery,2024-03,2128,grid,grid_electricity_NG,kWh,0.508,1081.024


In [ ]:
otootttt